In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.nn as pyg_nn
from torch_geometric.data import Data
from torch_geometric.transforms import KNNGraph

/public/home/zju12218076/miniconda3/envs/project/lib/python3.10/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.27' not found (required by /public/home/zju12218076/miniconda3/envs/project/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/public/home/zju12218076/miniconda3/envs/project/lib/python3.10/site-packages/torch_geometric/typing.py:124: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.27' not found (required by /public/home/zju12218076/miniconda3/envs/project/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "
/public/home/zju12218076/miniconda3/envs/project/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipyw

In [6]:
class DNA3DInteractionModel_V4_Softplus(nn.Module):
    """
    DNA三维交互预测模型 - 使用Softplus输出和泊松损失。

    此版本直接预测原始Hi-C值，无需手动log变换。

    关键修正:
    1.  **使用Softplus作为最终激活函数**: 保证输出非负且平滑，训练更稳定。
    2.  **无需数据log变换**: 直接使用原始Hi-C值作为训练目标。
    3.  **适配泊松损失**: 理论上最符合Hi-C数据的计数特性。
    """
    def __init__(self, *args, **kwargs):
        # 结构与V3完全相同
        super().__init__()
        # ... (此处省略与V3完全相同的__init__代码)
        node_feature_dim = kwargs.get('node_feature_dim')
        hidden_dim = kwargs.get('hidden_dim')
        pos_embedding_dim = kwargs.get('pos_embedding_dim', 16)
        num_gnn_layers = kwargs.get('num_gnn_layers', 4)
        num_final_predictor_layers = kwargs.get('num_final_predictor_layers', 2)
        dropout = kwargs.get('dropout', 0.1)

        self.input_proj = nn.Linear(node_feature_dim, hidden_dim)
        self.relative_pos_embedding = nn.Embedding(2048, pos_embedding_dim)
        self.gnn_layers = nn.ModuleList()
        for _ in range(num_gnn_layers):
            self.gnn_layers.append(pyg_nn.GATv2Conv(hidden_dim, hidden_dim // 4, heads=4))
        self.gnn_norm = nn.LayerNorm(hidden_dim)
        
        predictor_input_dim = hidden_dim * 2 + pos_embedding_dim
        predictor_layers = []
        for _ in range(num_final_predictor_layers):
            predictor_layers.extend([
                nn.Linear(predictor_input_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            predictor_input_dim = hidden_dim
        predictor_layers.append(nn.Linear(hidden_dim, 1))
        self.interaction_predictor = nn.Sequential(*predictor_layers)
        self.dropout = nn.Dropout(dropout)


    def forward(self, data: Data):
        # ... (此处省略与V3完全相同的前向传播逻辑，直到最后一步)
        x, edge_index = data.x, data.edge_index
        num_nodes = x.size(0)

        h = self.input_proj(x)
        h_gnn = h
        for gnn_layer in self.gnn_layers:
            h_gnn = gnn_layer(h_gnn, edge_index)
            h_gnn = F.relu(h_gnn)
            h_gnn = self.dropout(h_gnn)
        h_gnn = self.gnn_norm(h_gnn)
        
        h_i = h_gnn.unsqueeze(1).expand(-1, num_nodes, -1)
        h_j = h_gnn.unsqueeze(0).expand(num_nodes, -1, -1)
        
        pos = torch.arange(num_nodes, device=x.device)
        relative_pos = torch.abs(pos.unsqueeze(1) - pos.unsqueeze(0))
        relative_pos = torch.clamp(relative_pos, max=self.relative_pos_embedding.num_embeddings - 1)
        pos_embed = self.relative_pos_embedding(relative_pos)
        
        pair_features = torch.cat([h_i, h_j, pos_embed], dim=-1)
        
        # 得到对称的、激活前的 logits
        interaction_logits = self.interaction_predictor(pair_features).squeeze(-1)
        interaction_logits = (interaction_logits + interaction_logits.T) / 2.0
        
        # 返回 logits 和 softplus 激活后的值
        # 这样做可以灵活地选择损失函数
        return interaction_logits, F.softplus(interaction_logits)

In [8]:
NUM_NODES = 1024
NODE_FEATURE_DIM = 32
HIDDEN_DIM = 128
POS_EMBED_DIM = 16
K_NEIGHBORS = 10

In [12]:
mock_node_features = torch.randn(NUM_NODES, NODE_FEATURE_DIM)
mock_raw_hic_matrix = torch.randint(0, 2000, (NUM_NODES, NUM_NODES)).float()


In [14]:
data = Data(x=mock_node_features, y=mock_raw_hic_matrix) 
knn_transform = KNNGraph(k=K_NEIGHBORS, loop=True)
data.pos = torch.arange(NUM_NODES, dtype=torch.float).view(-1, 1)
data_with_edges = knn_transform(data)

In [34]:
model_v4 = DNA3DInteractionModel_V4_Softplus(
    node_feature_dim=NODE_FEATURE_DIM,
    hidden_dim=HIDDEN_DIM,
    pos_embedding_dim=POS_EMBED_DIM
)

In [37]:
criterion = nn.PoissonNLLLoss(log_input=False, full=True) 
optimizer = torch.optim.Adam(model_v4.parameters(), lr=0.001)


In [42]:
# model_v4.train()
# for epoch in range(150): # 实际训练需要更多轮次
#     optimizer.zero_grad()
    
#     # 前向传播，得到激活前的logits和激活后的预测值
#     predicted_logits, predicted_values = model_v4(data_with_edges)
    
#     # 计算损失：将logits和原始目标值送入泊松损失函数
#     loss = criterion(predicted_logits, data_with_edges.y)
    
#     loss.backward()
#     optimizer.step()
    
#     # 为了直观，我们可以计算一下预测值和真实值之间的MAE
#     mae = F.l1_loss(predicted_values, data_with_edges.y)
#     print(f"Epoch [{epoch+1}/10], Poisson Loss: {loss.item():.4f}, MAE: {mae.item():.2f}")

# print("训练完成!")

In [43]:
predicted_logits

tensor([[ 921.2485,  936.8181,  998.5116,  ..., 1045.5684,  988.2747,
         1049.7083],
        [ 936.8181,  894.3248,  973.6694,  ...,  901.9771,  999.8693,
         1015.5342],
        [ 998.5116,  973.6694,  966.3767,  ...,  962.0959,  973.1110,
          969.1907],
        ...,
        [1045.5684,  901.9771,  962.0959,  ...,  989.7141,  985.9531,
          953.8422],
        [ 988.2747,  999.8693,  973.1110,  ...,  985.9531,  878.5485,
          981.4778],
        [1049.7083, 1015.5342,  969.1907,  ...,  953.8422,  981.4778,
         1006.7402]], grad_fn=<DivBackward0>)

predicted_values

In [44]:
predicted_values

tensor([[ 921.2485,  936.8181,  998.5116,  ..., 1045.5684,  988.2747,
         1049.7083],
        [ 936.8181,  894.3248,  973.6694,  ...,  901.9771,  999.8693,
         1015.5342],
        [ 998.5116,  973.6694,  966.3767,  ...,  962.0959,  973.1110,
          969.1907],
        ...,
        [1045.5684,  901.9771,  962.0959,  ...,  989.7141,  985.9531,
          953.8422],
        [ 988.2747,  999.8693,  973.1110,  ...,  985.9531,  878.5485,
          981.4778],
        [1049.7083, 1015.5342,  969.1907,  ...,  953.8422,  981.4778,
         1006.7402]], grad_fn=<SoftplusBackward0>)

40960